# Question 1 extension: comparing inputs and training settings

This extends `../Question1/submission_model.ipynb`. I wanted to check whether distance or minutes remaining helps beyond the two coordinates so I tried comparing four input combinations and then try different learning rates and training lengths. The original task's interface takes two coordinates only and the three- and four-input models I made here are just extra experiments I did **for the love of the game lol**.

## 1. Read the actual CSV

The four possible inputs are `loc_x`, `loc_y`, `minutes_remaining`, and `shot_distance`. `shot_made_flag` is the answer. Since `shot_id` is simply a row identifier I just excluded it. Same as the original notebook, the cells print the columns and counts before training. All trials will be using the same supplied 25,000 labeled rows.

In [22]:
import csv
import json
import math
import random
from pathlib import Path

import torch
from torch import nn

for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / 'ICAIC_Outputs' / 'Question1').is_dir():
        OUTPUTS_ROOT = parent / 'ICAIC_Outputs'
        break
    if (parent / 'Question1').is_dir():
        OUTPUTS_ROOT = parent
        break

DATA_PATH = OUTPUTS_ROOT / 'Question1' / 'data_train.csv'
OUTPUT_DIR = OUTPUTS_ROOT / 'Question1-Extension'
OUTPUT_DIR.mkdir(exist_ok=True)
FEATURE_COLUMNS = ('loc_x', 'loc_y', 'minutes_remaining', 'shot_distance')
torch.set_num_threads(1)
with DATA_PATH.open(newline='', encoding='utf-8-sig') as file:
    reader = csv.DictReader(file)
    print('CSV columns:', reader.fieldnames)
    rows = list(reader)
features = torch.tensor([[float(row[name]) for name in FEATURE_COLUMNS] for row in rows])
labels = torch.tensor([[float(row['shot_made_flag'])] for row in rows])
print('Reading:', DATA_PATH)
print('Feature shape:', tuple(features.shape))
print('Made:', int(labels.sum()), '; missed:', len(labels) - int(labels.sum()))


CSV columns: ['loc_x', 'loc_y', 'minutes_remaining', 'shot_distance', 'shot_made_flag', 'shot_id']
Reading: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question1/data_train.csv
Feature shape: (25000, 4)
Made: 11186 ; missed: 13814


## 2. Separate selection

I keep the original 5,000-row holdout split with a 42 seed and then split the 20,000 development rows into 16,000 training and 4,000 validation rows with a 43 seed. It essentially does the same thing as the original notebook.

In [23]:
def split_rows(indices, labels, seed):
    rng = random.Random(seed)
    answers = labels.reshape(-1).tolist()
    learning, checking = [], []
    for label in sorted(set(answers)):
        group = [index for index in indices if answers[index] == label]
        rng.shuffle(group)
        count = round(0.2 * len(group))
        checking.extend(group[:count])
        learning.extend(group[count:])
    rng.shuffle(learning)
    rng.shuffle(checking)
    return torch.tensor(learning), torch.tensor(checking)

development, holdout = split_rows(list(range(len(labels))), labels, 42)
training, validation = split_rows(development.tolist(), labels, 43)
print('Training:', len(training), '; validation:', len(validation), '; holdout:', len(holdout))


def scale_for(indices, columns):
    values = features[:, columns]
    mean = values[indices].mean(dim=0)
    std = values[indices].std(dim=0, unbiased=False)
    return (values - mean) / std, mean, std


Training: 16000 ; validation: 4000 ; holdout: 5000


# 3. Keep the hidden layers the same

The model is still an MLP with hidden widths 8 and 8, ReLU activations, and one Sigmoid output so only the first layer's input size changes. The general shape of the model will look like as follows:

```text
2, 3, or 4 inputs → Linear(inputs,8) → ReLU → Linear(8,8) → ReLU → Linear(8,1) → Sigmoid
```

In [24]:
class Model(nn.Module):
    def __init__(self, input_size=2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(8, 8)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.sigmoid(self.fc3(x))

@torch.no_grad()
def evaluate(model, features, labels):
    probabilities = model(features)
    correct = ((probabilities >= 0.5).float() == labels).sum().item()
    return {'accuracy': correct / len(labels), 'bce': nn.BCELoss()(probabilities, labels).item(),
            'correct': correct, 'total': len(labels)}


def fit(x, y, learning_rate, epochs, validation_data=None):
    torch.manual_seed(43)
    model = Model(x.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.0001)
    generator = torch.Generator().manual_seed(1043)
    checkpoints = []
    for epoch in range(1, epochs + 1):
        rate = learning_rate * (0.1 + 0.9 * (1 + math.cos(math.pi * (epoch - 1) / 400)) / 2)
        optimizer.param_groups[0]['lr'] = rate
        order = torch.randperm(len(x), generator=generator)
        for start in range(0, len(x), 1024):
            batch = order[start:start + 1024]
            optimizer.zero_grad()
            loss = nn.BCELoss()(model(x[batch]), y[batch])
            loss.backward()
            optimizer.step()
        if validation_data is not None and epoch in (90, 180, 270):
            metrics = evaluate(model, *validation_data)
            checkpoints.append({'epoch': epoch, **metrics})
    model.eval()
    return model, checkpoints


## 4. Compare the four input combinations

First, I compared all of the input combinations at the original learning rate of 0.01. Each trial runs for 270 epochs and checks validation at epochs 90, 180, and 270. All of these checkpoints let me compare training length without starting three separate runs.

The next cell repeats the same comparison at rates 0.005 and 0.02 which makes a sum of 12 training runs and 36 validation measurements in total.

In [25]:
input_options = [
    ('Coordinates', [0, 1]),
    ('Coordinates + distance', [0, 1, 3]),
    ('Coordinates + minutes', [0, 1, 2]),
    ('All four inputs', [0, 1, 2, 3]),
]
results = []


def try_rate(learning_rate):
    for name, columns in input_options:
        scaled, _, _ = scale_for(training, columns)
        _, checkpoints = fit(scaled[training], labels[training], learning_rate, 270,
                             validation_data=(scaled[validation], labels[validation]))
        for measurement in checkpoints:
            results.append({'name': name, 'columns': columns, 'learning_rate': learning_rate, **measurement})
        best = max(checkpoints, key=lambda row: (row['accuracy'], -row['bce']))
        print(f"{name}: rate={learning_rate}; best epoch={best['epoch']}; "
              f"validation={best['accuracy']:.2%}; BCE={best['bce']:.6f}", flush=True)


try_rate(0.01)


Coordinates: rate=0.01; best epoch=270; validation=60.15%; BCE=0.667241
Coordinates + distance: rate=0.01; best epoch=270; validation=60.00%; BCE=0.667853
Coordinates + minutes: rate=0.01; best epoch=270; validation=59.98%; BCE=0.667237
All four inputs: rate=0.01; best epoch=90; validation=59.98%; BCE=0.668194


In [26]:
for rate in (0.005, 0.02):
    print('\nTrying learning rate:', rate)
    try_rate(rate)

print('\nFull validation comparison')
print('Inputs | learning rate | epochs | validation accuracy | BCE')
for row in results:
    print(f"{row['name']} | {row['learning_rate']} | {row['epoch']} | {row['accuracy']:.2%} | {row['bce']:.6f}")
with (OUTPUT_DIR / 'validation_results.csv').open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)
selected = max(results, key=lambda row: (row['accuracy'], -row['bce']))
selected_two = max((row for row in results if len(row['columns']) == 2),
                   key=lambda row: (row['accuracy'], -row['bce']))
print('Best overall by validation:', selected)
print('Best two-input recipe by validation:', selected_two)



Trying learning rate: 0.005
Coordinates: rate=0.005; best epoch=90; validation=59.82%; BCE=0.668188
Coordinates + distance: rate=0.005; best epoch=270; validation=59.95%; BCE=0.667747
Coordinates + minutes: rate=0.005; best epoch=90; validation=59.80%; BCE=0.667607
All four inputs: rate=0.005; best epoch=270; validation=60.00%; BCE=0.667689

Trying learning rate: 0.02
Coordinates: rate=0.02; best epoch=90; validation=59.92%; BCE=0.667615
Coordinates + distance: rate=0.02; best epoch=90; validation=60.12%; BCE=0.667993
Coordinates + minutes: rate=0.02; best epoch=270; validation=59.98%; BCE=0.666379
All four inputs: rate=0.02; best epoch=270; validation=59.95%; BCE=0.667577

Full validation comparison
Inputs | learning rate | epochs | validation accuracy | BCE
Coordinates | 0.01 | 90 | 59.95% | 0.667639
Coordinates | 0.01 | 180 | 59.98% | 0.667880
Coordinates | 0.01 | 270 | 60.15% | 0.667241
Coordinates + distance | 0.01 | 90 | 60.00% | 0.668052
Coordinates + distance | 0.01 | 180 | 59

## 5. Check the selected recipes and export

I then refit the best two-input recipe and the best overall recipe on the 20,000 development rows before the local check. After that, I trained fresh final models on all 25,000 labeled rows. I already arranged the files so that the selected exports are stored in `selected_two_inputs` and `selected_overall`; their `input_columns.json` files document the required column order.


In [27]:
checked = {}
exports = {}
for label, recipe in [('selected_two_inputs', selected_two), ('selected_overall', selected)]:
    key = (tuple(recipe['columns']), recipe['learning_rate'], recipe['epoch'])
    if key not in checked:
        scaled, _, _ = scale_for(development, recipe['columns'])
        evaluation_model, _ = fit(scaled[development], labels[development], recipe['learning_rate'], recipe['epoch'])
        checked[key] = evaluate(evaluation_model, scaled[holdout], labels[holdout])
    print(label, 'local holdout:', checked[key])

    scaled, mean, std = scale_for(list(range(len(features))), recipe['columns'])
    model, _ = fit(scaled, labels, recipe['learning_rate'], recipe['epoch'])
    exported_model = Model(len(recipe['columns']))
    exported_model.load_state_dict(model.state_dict())
    with torch.no_grad():
        weights = model.fc1.weight / std
        exported_model.fc1.weight.copy_(weights)
        exported_model.fc1.bias.copy_(model.fc1.bias - (weights * mean).sum(dim=1))
        torch.testing.assert_close(exported_model(features[:, recipe['columns']]), model(scaled), rtol=1e-5, atol=1e-6)
    folder = OUTPUT_DIR / label
    folder.mkdir(exist_ok=True)
    torch.save(exported_model.state_dict(), folder / 'submission_dic.pth')
    columns = [FEATURE_COLUMNS[index] for index in recipe['columns']]
    (folder / 'input_columns.json').write_text(json.dumps(columns, indent=2) + '\n')
    exports[label] = {'recipe': recipe, 'holdout': checked[key], 'input_columns': columns}


selected_two_inputs local holdout: {'accuracy': 0.6036, 'bce': 0.6648858189582825, 'correct': 3018, 'total': 5000}
selected_overall local holdout: {'accuracy': 0.6036, 'bce': 0.6648858189582825, 'correct': 3018, 'total': 5000}


In [28]:
model_template = """from torch import nn

class Model(nn.Module):
    def __init__(self, input_size=2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 8)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(8, 8)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.sigmoid(self.fc3(x))
"""

import importlib.util
for label, data in exports.items():
    folder = OUTPUT_DIR / label
    definition = model_template.replace('input_size=2', f"input_size={len(data['input_columns'])}")
    (folder / 'submission_model.py').write_text(definition, encoding='utf-8')
    spec = importlib.util.spec_from_file_location(label, folder / 'submission_model.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    restored = module.Model()
    restored.load_state_dict(torch.load(folder / 'submission_dic.pth', map_location='cpu', weights_only=True))
    with torch.no_grad():
        probabilities = restored(features[:2, data['recipe']['columns']]).squeeze(1)
    print(label, 'first two probabilities:', probabilities.tolist(), '; actual:', labels[:2].squeeze(1).int().tolist())

summary = {'exports': exports, 'selection_metric': 'validation accuracy, then lower BCE',
           'split_sizes': {'training': len(training), 'validation': len(validation), 'holdout': len(holdout)},
           'holdout_previously_inspected': True}
(OUTPUT_DIR / 'experiment_results.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Saved comparisons and exports in:', OUTPUT_DIR)


selected_two_inputs first two probabilities: [0.6353702545166016, 0.485137403011322] ; actual: [1, 0]
selected_overall first two probabilities: [0.6353702545166016, 0.485137403011322] ; actual: [1, 0]
Saved comparisons and exports in: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question1-Extension


## What the comparison means

The validation results show which recipe worked best among the settings tried and it showed that **more inputs do not automatically improve accuracy**. So, only using the 2 input `loc_x` and `loc_y` was the right and most optimal choice after all.


## Measured result

| Inputs | Learning rate | Epochs | Best validation accuracy |
| --- | ---: | ---: | ---: |
| Coordinates | 0.01 | 270 | 60.15% |
| Coordinates + distance | 0.02 | 90 | 60.12% |
| Coordinates + minutes | 0.02 | 270 | 59.98% |
| All four inputs | 0.005 | 270 | 60.00% |

As what the values reflect, the two-coordinate recipe won this validation comparison with a learning rate of 0.01 with 270 total epochs. Its local holdout accuracy was **60.36%**, which was the same as the original notebook. Adding the extra columns did not improve the selected model in these trials and some validation differences are only one or a few shots so they should not be treated as strong evidence that a column is always unhelpful.
